To run this, press "*Runtime*" and press "*Run all*" on a **free** Tesla T4 Google Colab instance!
<div class="align-center">
<a href="https://unsloth.ai/"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
<a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord button.png" width="145"></a>
<a href="https://unsloth.ai/docs/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a> Join Discord if you need help + ⭐ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐
</div>

To install Unsloth on your local device, follow [our guide](https://unsloth.ai/docs/get-started/install). This notebook is licensed [LGPL-3.0](https://github.com/unslothai/notebooks?tab=LGPL-3.0-1-ov-file#readme).

You will learn how to do [data prep](#Data), how to [train](#Train), how to [run the model](#Inference), & how to save it

### News

Train MoEs - DeepSeek, GLM, Qwen and gpt-oss 12x faster with 35% less VRAM. [Blog](https://unsloth.ai/docs/new/faster-moe)

You can now train embedding models 1.8-3.3x faster with 20% less VRAM. [Blog](https://unsloth.ai/docs/new/embedding-finetuning)

Ultra Long-Context Reinforcement Learning is here with 7x more context windows! [Blog](https://unsloth.ai/docs/new/grpo-long-context)

3x faster LLM training with 30% less VRAM and 500K context. [3x faster](https://unsloth.ai/docs/new/3x-faster-training-packing) • [500K Context](https://unsloth.ai/docs/new/500k-context-length-fine-tuning)

New in Reinforcement Learning: [FP8 RL](https://unsloth.ai/docs/new/fp8-reinforcement-learning) • [Vision RL](https://unsloth.ai/docs/new/vision-reinforcement-learning-vlm-rl) • [Standby](https://unsloth.ai/docs/basics/memory-efficient-rl) • [gpt-oss RL](https://unsloth.ai/docs/new/gpt-oss-reinforcement-learning)

Visit our docs for all our [model uploads](https://unsloth.ai/docs/get-started/unsloth-model-catalog) and [notebooks](https://unsloth.ai/docs/get-started/unsloth-notebooks).

### Installation

In [ ]:
%%capture
import importlib.util
import os
import platform
import subprocess
import sys


def uv_pip_install(*args):
    subprocess.check_call([sys.executable, "-m", "uv", "pip", "install", *args])


subprocess.check_call([sys.executable, "-m", "pip", "install", "--upgrade", "-qqq", "uv"])

is_colab = "COLAB_" in "".join(os.environ.keys())
is_linux = platform.system() == "Linux"

if importlib.util.find_spec("torch") is None or is_colab:
    try:
        import numpy
        import PIL
        numpy_pin = f"numpy=={numpy.__version__}"
        pillow_pin = f"pillow=={PIL.__version__}"
    except Exception:
        numpy_pin = "numpy"
        pillow_pin = "pillow"

    base_packages = [
        "-qqq",
        "torch==2.8.0",
        numpy_pin,
        pillow_pin,
        "torchvision",
        "bitsandbytes",
        "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo",
        "unsloth[base] @ git+https://github.com/unslothai/unsloth",
    ]
    if is_linux:
        base_packages.extend(["triton>=3.3.0", "xformers==0.0.32.post2"])

    uv_pip_install(*base_packages)
elif importlib.util.find_spec("unsloth") is None:
    uv_pip_install("-qqq", "unsloth")

uv_pip_install("--upgrade", "--no-deps", "tokenizers", "trl==0.22.2", "unsloth", "unsloth_zoo")
try:
    uv_pip_install("transformers==5.2.0")
except subprocess.CalledProcessError:
    uv_pip_install("transformers>=5.0.0,<6")

subprocess.run([sys.executable, "-m", "uv", "pip", "uninstall", "-y", "torchcodec"], check=False)

if "torchcodec" in sys.modules:
    del sys.modules["torchcodec"]

if is_linux:
    uv_pip_install("--no-build-isolation", "flash-linear-attention", "causal_conv1d==1.6.0")

required_modules = ("torch", "transformers", "trl", "unsloth")
missing_modules = [name for name in required_modules if importlib.util.find_spec(name) is None]
if missing_modules:
    raise ImportError(f"Missing required modules after install: {missing_modules}")


### Unsloth

In [ ]:
import os
import sys

sys.modules["torchcodec"] = None
os.environ.setdefault("HF_HUB_DISABLE_TELEMETRY", "1")

import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "Notebook fine-tuning nay can GPU CUDA. Tren CPU, 4-bit loading va training voi Unsloth se rat de loi."
    )

import unsloth
from unsloth import FastLanguageModel

model_name = os.environ.get("ERIS_MODEL_NAME", "unsloth/Qwen3.5-4B")
max_seq_length = int(os.environ.get("ERIS_MAX_SEQ_LENGTH", "384"))
load_in_4bit = os.environ.get("ERIS_LOAD_IN_4BIT", "1") != "0"
device = "cuda"

print(f"Loading model: {model_name}")
print(f"max_seq_length={max_seq_length}, load_in_4bit={load_in_4bit}")

try:
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = model_name,
        max_seq_length = max_seq_length,
        dtype = None,
        load_in_4bit = load_in_4bit,
    )
except Exception as exc:
    raise RuntimeError(
        f"Khong tai duoc model {model_name}. Neu repo doi ten hoac mang bi chan, hay set ERIS_MODEL_NAME truoc khi chay cell nay."
    ) from exc


We now add LoRA adapters for parameter efficient finetuning - this allows us to only efficiently train 1% of all parameters.

**[NEW]** We also support finetuning ONLY the vision part of the model, or ONLY the language part. Or you can select both! You can also select to finetune the attention or the MLP layers!

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 32,  # Rank 32: đủ giàu để học phong cách nói chuyện mà không quá nặng
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 32,  # Giữ alpha 32 để cập nhật đủ mạnh nhưng ổn định
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

<a name="Data"></a>
### Data Prep
We'll be using a sampled dataset of handwritten maths formulas. The goal is to convert these images into a computer readable form - ie in LaTeX form, so we can render it. This can be very useful for complex formulas.

You can access the dataset [here](https://huggingface.co/datasets/unsloth/LaTeX_OCR). The full dataset is [here](https://huggingface.co/datasets/linxy/LaTeX_OCR).

In [ ]:
import os
import sys
from pathlib import Path
import numpy as np
from unsloth.chat_templates import get_chat_template

module_candidates = [Path("full_train_dataset.py"), Path("src/full_train_dataset.py")]
module_path = next((path for path in module_candidates if path.exists()), None)
if module_path is None:
    raise FileNotFoundError("Khong tim thay full_train_dataset.py o cwd hien tai hoac trong thu muc src.")

module_dir = str(module_path.parent.resolve())
if module_dir not in sys.path:
    sys.path.insert(0, module_dir)

from full_train_dataset import build_messages, build_text_dataset, find_dataset_path

dataset_path = find_dataset_path([Path("dataset.jsonl"), Path("src/dataset.jsonl")])

tokenizer = get_chat_template(
    tokenizer,
    chat_template = "chatml",
)

print(f"Reading dataset from: {dataset_path}")
dataset, dataset_report = build_text_dataset(dataset_path, tokenizer)

dataset_limit = int(os.environ.get("ERIS_DATASET_LIMIT", "0"))
if dataset_limit > 0:
    dataset = dataset.select(range(min(dataset_limit, len(dataset))))
    print(f"Using the first {len(dataset)} rows because ERIS_DATASET_LIMIT={dataset_limit}")

base_tokenizer = getattr(tokenizer, "tokenizer", tokenizer)


def count_tokens(text):
    try:
        return len(base_tokenizer.encode(text, add_special_tokens=True))
    except TypeError:
        return len(base_tokenizer.encode(text))


token_sample_size = min(len(dataset), int(os.environ.get("ERIS_TOKEN_STATS_SAMPLE", "512")))
token_counts = [count_tokens(text) for text in dataset["text"][:token_sample_size]]
p90_tokens = int(np.percentile(token_counts, 90)) if token_counts else max_seq_length
suggested_length = int(np.ceil(p90_tokens / 64) * 64) if token_counts else max_seq_length

print(f"Loaded {dataset_report['loaded_rows']} training rows from {dataset_path}")
print(f"Skipped {dataset_report['skipped_rows']} malformed rows during normalization")
print(f"Detected line encodings: {dataset_report.get('encoding_counts', {})}")
for example in dataset_report["examples"]:
    print(f"- line {example['line']}: {example['reason']}")

if token_counts:
    print(f"Token stats sample size: {token_sample_size}")
    print(f"Mean sampled length: {np.mean(token_counts):.2f}")
    print(f"90th percentile sampled length: {np.percentile(token_counts, 90):.2f}")
    print(f"Max sampled length: {np.max(token_counts)}")
    print(f"Suggested max_seq_length: {suggested_length}")
    if suggested_length + 64 < max_seq_length:
        print(f"Tip: set ERIS_MAX_SEQ_LENGTH={suggested_length} before loading the model to reduce training time.")


<a name="Train"></a>
### Train the model
The data cell now prints a quick token-length estimate before training so you can judge whether the run will fit inside a free Colab session.

For a safer free-tier run, the training cell defaults to `1` epoch, enables packing and length grouping when the installed TRL version supports them, and disables intermediate checkpoint saving by default. Override these with `ERIS_*` environment variables before rerunning the training cell.

In [ ]:
import os
from inspect import signature
from trl import SFTTrainer, SFTConfig
from transformers import DataCollatorForSeq2Seq
from unsloth.chat_templates import train_on_responses_only
import math

FastLanguageModel.for_training(model)

per_device_train_batch_size = int(os.environ.get("ERIS_PER_DEVICE_TRAIN_BATCH_SIZE", "2"))
gradient_accumulation_steps = int(os.environ.get("ERIS_GRADIENT_ACCUMULATION_STEPS", "4"))
num_train_epochs = float(os.environ.get("ERIS_NUM_TRAIN_EPOCHS", "1"))
max_steps_override = int(os.environ.get("ERIS_MAX_STEPS", "0"))
learning_rate = float(os.environ.get("ERIS_LEARNING_RATE", "5e-5"))
packing = os.environ.get("ERIS_PACKING", "1") != "0"
group_by_length = os.environ.get("ERIS_GROUP_BY_LENGTH", "1") != "0"
dataloader_num_workers = int(os.environ.get("ERIS_DATALOADER_NUM_WORKERS", "2"))
save_strategy = os.environ.get("ERIS_SAVE_STRATEGY", "no")
effective_batch_size = per_device_train_batch_size * gradient_accumulation_steps
steps_per_epoch = math.ceil(len(dataset) / effective_batch_size)
total_training_steps = max_steps_override if max_steps_override > 0 else math.ceil(steps_per_epoch * num_train_epochs)
warmup_steps = max(50, math.ceil(total_training_steps * 0.03))
logging_steps = max(1, steps_per_epoch // 100)
supports_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
padding_tokenizer = getattr(tokenizer, "tokenizer", tokenizer)

print(f"Train rows: {len(dataset)}")
print(f"Effective batch size: {effective_batch_size}")
print(f"Optimizer steps per epoch: {steps_per_epoch}")
print(f"Total optimizer steps: {total_training_steps}")
print(f"Warmup steps: {warmup_steps}")
print(f"Logging every {logging_steps} optimizer steps")
print(f"Packing enabled: {packing}")
print(f"Group by length: {group_by_length}")
if max_steps_override > 0:
    print(f"Max steps override: {max_steps_override}")
if total_training_steps > 1200:
    print("Warning: this run is likely to be tight on free Colab. Consider ERIS_NUM_TRAIN_EPOCHS=1, ERIS_MAX_STEPS=600, or ERIS_DATASET_LIMIT.")

sft_config_signature = signature(SFTConfig.__init__)
sft_config_kwargs = dict(
    per_device_train_batch_size = per_device_train_batch_size,
    gradient_accumulation_steps = gradient_accumulation_steps,
    warmup_steps = warmup_steps,
    num_train_epochs = num_train_epochs,
    learning_rate = learning_rate,
    fp16 = not supports_bf16,
    bf16 = supports_bf16,
    logging_steps = logging_steps,
    optim = "adamw_8bit",
    weight_decay = 0.01,
    lr_scheduler_type = "cosine",
    seed = 3407,
    output_dir = "outputs",
    report_to = "none",
)

optional_config_kwargs = {
    "max_steps": max_steps_override if max_steps_override > 0 else -1,
    "save_strategy": save_strategy,
    "group_by_length": group_by_length,
    "dataloader_num_workers": dataloader_num_workers,
    "packing": packing,
}
for key, value in optional_config_kwargs.items():
    if key in sft_config_signature.parameters:
        sft_config_kwargs[key] = value

trainer_kwargs = dict(
    model = model,
    train_dataset = dataset,
    max_seq_length = max_seq_length,
    data_collator = DataCollatorForSeq2Seq(tokenizer = padding_tokenizer),
    args = SFTConfig(**sft_config_kwargs),
)

sft_signature = signature(SFTTrainer.__init__)
if "tokenizer" in sft_signature.parameters:
    trainer_kwargs["tokenizer"] = tokenizer
elif "processing_class" in sft_signature.parameters:
    trainer_kwargs["processing_class"] = tokenizer
else:
    raise TypeError("Khong tim thay tham so tokenizer/processing_class tu SFTTrainer")

if "dataset_text_field" in sft_signature.parameters:
    trainer_kwargs["dataset_text_field"] = "text"
elif "formatting_func" in sft_signature.parameters:
    trainer_kwargs["formatting_func"] = lambda sample: sample["text"]

if "packing" in sft_signature.parameters and "packing" not in sft_config_kwargs:
    trainer_kwargs["packing"] = packing

trainer = SFTTrainer(**trainer_kwargs)

try:
    trainer = train_on_responses_only(
        trainer,
        instruction_part = "<|im_start|>user\n",
        response_part = "<|im_start|>assistant\n",
    )
except Exception as exc:
    print(f"Responses-only masking khong bat duoc: {exc}")
    print("Trainer se tiep tuc voi masking mac dinh cua TRL.")

trainer_stats = trainer.train()


In [ ]:
print("Useful Colab overrides for the next run:")
print("- ERIS_NUM_TRAIN_EPOCHS=1 for a full but safer run")
print("- ERIS_MAX_STEPS=600 to cap the session length")
print(f"- ERIS_MAX_SEQ_LENGTH={suggested_length} if your sampled token stats stay similar")
print("- ERIS_DATASET_LIMIT=2000 for a quick smoke test")
print("- ERIS_MODEL_NAME=unsloth/Qwen3.5-1.7B for a much faster baseline")


If the sampled token statistics suggest a much smaller `max_seq_length`, rerun the model-loading cell after setting `ERIS_MAX_SEQ_LENGTH` so the smaller context length actually takes effect.

In [ ]:
# @title Show current memory stats
gpu_stats = None
start_gpu_memory = 0.0
max_memory = 0.0

if torch.cuda.is_available():
    gpu_stats = torch.cuda.get_device_properties(0)
    start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
    max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
    print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
    print(f"{start_gpu_memory} GB of memory reserved.")
else:
    print("CUDA not available. Memory stats are only collected on GPU runs.")


In [ ]:
# @title Show final memory and time stats
if "trainer_stats" not in locals():
    print("trainer_stats chua ton tai. Hay chay cell train truoc.")
elif not torch.cuda.is_available():
    print("CUDA not available. Skipping final GPU memory summary.")
else:
    used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
    used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
    used_percentage = round(used_memory / max_memory * 100, 3) if max_memory else 0.0
    lora_percentage = round(used_memory_for_lora / max_memory * 100, 3) if max_memory else 0.0
    print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
    print(f"{round(trainer_stats.metrics['train_runtime'] / 60, 2)} minutes used for training.")
    print(f"Peak reserved memory = {used_memory} GB.")
    print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
    print(f"Peak reserved memory % of max memory = {used_percentage} %.")
    print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")


<a name="Inference"></a>
### Inference
Let's run the model! You can change the instruction and input - leave the output blank!

We use `min_p = 0.1` and `temperature = 1.5`. Read this [Tweet](https://x.com/menhguin/status/1826132708508213629) for more information on why.

In [ ]:
FastLanguageModel.for_inference(model)
base_tokenizer = getattr(tokenizer, "tokenizer", tokenizer)

machine_context = {
    "time": "02:17",
    "active_app": "VS Code",
    "idle_seconds": 640,
    "recent_events": ["build_failed", "same_song_loop_5x"],
}

messages = build_messages(
    [
        {"role": "system", "content": "Ban la ERIS, mot AI co khi chat INFJ: tinh te, giau dong cam, kin dao, doi khi lanh va nghiem nhung khong tan nhan. Goi nguoi dung la Ngai va xung toi."},
        {"role": "user", "content": "Toi van on."},
    ],
    machine_context = machine_context,
)

input_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(text=[input_text], return_tensors="pt").to(device)
from transformers import TextStreamer
text_streamer = TextStreamer(base_tokenizer, skip_prompt=True)
_ = model.generate(**inputs, streamer=text_streamer, max_new_tokens=128, use_cache=True, temperature=0.3)


<a name="Save"></a>
### Saving, loading finetuned models
To save the final model as LoRA adapters, either use Hugging Face's `push_to_hub` for an online save or `save_pretrained` for a local save.

**[NOTE]** This ONLY saves the LoRA adapters, and not the full model. To save to 16bit or GGUF, scroll down!

In [ ]:
model.save_pretrained("qwen_lora")  # Local saving
tokenizer.save_pretrained("qwen_lora")
# model.push_to_hub("your_name/qwen_lora", token = "YOUR_HF_TOKEN") # Online saving
# tokenizer.push_to_hub("your_name/qwen_lora", token = "YOUR_HF_TOKEN") # Online saving

Now if you want to load the LoRA adapters we just saved for inference, set `False` to `True`:

In [ ]:
if False:
    import unsloth
    from unsloth import FastLanguageModel
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = "qwen_lora",
        load_in_4bit = True,
    )
    FastLanguageModel.for_inference(model)

base_tokenizer = getattr(tokenizer, "tokenizer", tokenizer)
machine_context = {
    "time": "09:42",
    "active_app": "Chrome",
    "idle_seconds": 28,
    "recent_events": ["calendar_opened", "meeting_ended", "switched_tabs_14x"],
}

messages = build_messages(
    [
        {"role": "system", "content": "Ban la ERIS, mot AI co khi chat INFJ: tinh te, giau dong cam, kin dao, doi khi lanh va nghiem nhung khong tan nhan. Goi nguoi dung la Ngai va xung toi."},
        {"role": "user", "content": "Toi vua hop xong nhung dau van roi."},
    ],
    machine_context = machine_context,
)

input_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(
    text = [input_text],
    return_tensors = "pt",
    padding = True,
).to(device)
from transformers import TextStreamer
text_streamer = TextStreamer(base_tokenizer, skip_prompt=True)
_ = model.generate(**inputs, streamer=text_streamer, max_new_tokens=128, use_cache=True, temperature=0.3)


### Saving to float16 for VLLM

We also support saving to `float16` directly. Select `merged_16bit` for float16. Use `push_to_hub_merged` to upload to your Hugging Face account! You can go to https://huggingface.co/settings/tokens for your personal tokens. See [our docs](https://unsloth.ai/docs/basics/inference-and-deployment) for more deployment options.

In [ ]:
RUN_MERGED_EXPORT = False
RUN_GGUF_EXPORT = True
RUN_LLAMA_CPP_MANUAL_BUILD = False
GGUF_OUTPUT_DIR = os.environ.get("ERIS_GGUF_OUTPUT_DIR", "qwen_finetune")
GGUF_QUANTIZATION = os.environ.get("ERIS_GGUF_QUANTIZATION", "q4_k_m")

print("Optional export flags:")
print(f"- RUN_MERGED_EXPORT = {RUN_MERGED_EXPORT}")
print(f"- RUN_GGUF_EXPORT = {RUN_GGUF_EXPORT}")
print(f"- RUN_LLAMA_CPP_MANUAL_BUILD = {RUN_LLAMA_CPP_MANUAL_BUILD}")
print(f"- GGUF_OUTPUT_DIR = {GGUF_OUTPUT_DIR}")
print(f"- GGUF_QUANTIZATION = {GGUF_QUANTIZATION}")

if RUN_MERGED_EXPORT:
    model.save_pretrained_merged("unsloth_finetune", tokenizer)
else:
    print("Skipping merged 16-bit export. Set RUN_MERGED_EXPORT = True to enable.")


### GGUF / llama.cpp Conversion
To save to `GGUF` / `llama.cpp`, we support it natively now! We clone `llama.cpp` and we default save it to `q8_0`. We allow all methods like `q4_k_m`. Use `save_pretrained_gguf` for local saving and `push_to_hub_gguf` for uploading to HF.

Some supported quant methods (full list on our [docs page](https://unsloth.ai/docs/basics/inference-and-deployment/saving-to-gguf)):
* `q8_0` - Fast conversion. High resource use, but generally acceptable.
* `q4_k_m` - Recommended. Uses Q6_K for half of the attention.wv and feed_forward.w2 tensors, else Q4_K.
* `q5_k_m` - Recommended. Uses Q6_K for half of the attention.wv and feed_forward.w2 tensors, else Q5_K.

[**NEW**] To finetune and auto export to Ollama, try our [Ollama notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3_(8B)-Ollama.ipynb)

In [ ]:
if RUN_GGUF_EXPORT:
    model.save_pretrained_gguf(GGUF_OUTPUT_DIR, tokenizer, quantization_method = GGUF_QUANTIZATION)
else:
    print("Skipping GGUF export. Set RUN_GGUF_EXPORT = True in the previous cell to enable.")


Để thực hiện việc này, chúng ta cần tìm đường dẫn đến công cụ `quantize` của `llama.cpp`. Sau đó, sử dụng nó để chuyển đổi mô hình.

In [ ]:
import os
from pathlib import Path

llama_cpp_root = Path(os.environ.get("ERIS_LLAMA_CPP_PATH", "llama.cpp")).resolve()
quantize_candidates = [
    llama_cpp_root / "build" / "bin" / "llama-quantize",
    llama_cpp_root / "build" / "bin" / "llama-quantize.exe",
    llama_cpp_root / "build" / "bin" / "Release" / "llama-quantize.exe",
    llama_cpp_root / "build" / "bin" / "Release" / "llama-quantize",
    llama_cpp_root / "quantize",
]

if not RUN_LLAMA_CPP_MANUAL_BUILD:
    print("Skipping llama.cpp helper cells. Set RUN_LLAMA_CPP_MANUAL_BUILD = True to enable them.")
elif not llama_cpp_root.exists():
    print(f"Khong tim thay llama.cpp tai {llama_cpp_root}. Hay clone repo va set ERIS_LLAMA_CPP_PATH neu can.")
else:
    print(f"llama.cpp root: {llama_cpp_root}")
    existing_quantize = [path for path in quantize_candidates if path.exists()]
    if existing_quantize:
        print(f"Found quantize binary: {existing_quantize[0]}")
    else:
        print("Chua tim thay quantize binary. Neu can, hay chay cell build ben duoi.")


In [ ]:
import os
import shutil
import subprocess
from pathlib import Path

if not RUN_LLAMA_CPP_MANUAL_BUILD:
    print("Skipping llama.cpp build. Set RUN_LLAMA_CPP_MANUAL_BUILD = True to enable.")
else:
    llama_cpp_root = Path(os.environ.get("ERIS_LLAMA_CPP_PATH", "llama.cpp")).resolve()
    if not llama_cpp_root.exists():
        raise FileNotFoundError(f"Khong tim thay llama.cpp tai {llama_cpp_root}")

    build_dir = llama_cpp_root / "build"
    shutil.rmtree(build_dir, ignore_errors=True)
    build_dir.mkdir(parents=True, exist_ok=True)

    cmake_jobs = os.environ.get("ERIS_CMAKE_JOBS", "2")
    subprocess.check_call(["cmake", "..", "-DGGML_CPU=ON"], cwd=build_dir)
    subprocess.check_call(["cmake", "--build", ".", "--config", "Release", "-j", cmake_jobs], cwd=build_dir)


In [ ]:
import os
import subprocess
from pathlib import Path

if not RUN_LLAMA_CPP_MANUAL_BUILD:
    print("Skipping manual quantize helper. Set RUN_LLAMA_CPP_MANUAL_BUILD = True to enable.")
else:
    llama_cpp_root = Path(os.environ.get("ERIS_LLAMA_CPP_PATH", "llama.cpp")).resolve()
    quantize_candidates = [
        llama_cpp_root / "build" / "bin" / "llama-quantize",
        llama_cpp_root / "build" / "bin" / "llama-quantize.exe",
        llama_cpp_root / "build" / "bin" / "Release" / "llama-quantize.exe",
        llama_cpp_root / "build" / "bin" / "Release" / "llama-quantize",
        llama_cpp_root / "quantize",
    ]
    quantize_tool = next((path for path in quantize_candidates if path.exists()), None)
    if quantize_tool is None:
        raise FileNotFoundError("Khong tim thay llama-quantize. Hay build llama.cpp truoc.")

    gguf_dir = Path(os.environ.get("ERIS_GGUF_DIR", "qwen_finetune")).resolve()
    gguf_candidates = list(gguf_dir.glob("*.F16.gguf")) + list(gguf_dir.glob("*f16*.gguf")) + list(gguf_dir.glob("*.gguf"))
    input_f16 = next((path for path in gguf_candidates if path.exists()), None)
    if input_f16 is None:
        raise FileNotFoundError(f"Khong tim thay file GGUF nao trong {gguf_dir}")

    if ".F16.gguf" in input_f16.name:
        output_q4 = input_f16.with_name(input_f16.name.replace(".F16.gguf", ".Q4_K_M.gguf"))
    else:
        output_q4 = input_f16.with_name(f"{input_f16.stem}.Q4_K_M.gguf")

    subprocess.check_call([str(quantize_tool), str(input_f16), str(output_q4), "q4_K_M"])
    print(f"Wrote quantized model to {output_q4}")


And we're done! If you have any questions on Unsloth, we have a [Discord](https://discord.gg/unsloth) channel! If you find any bugs or want to keep updated with the latest LLM stuff, or need help, join projects etc, feel free to join our Discord!

Some other resources:
1. Looking to use Unsloth locally? Read our [Installation Guide](https://unsloth.ai/docs/get-started/install) for details on installing Unsloth on Windows, Docker, AMD, Intel GPUs.
2. Learn how to do Reinforcement Learning with our [RL Guide and notebooks](https://unsloth.ai/docs/get-started/reinforcement-learning-rl-guide).
3. Read our guides and notebooks for [Text-to-speech (TTS)](https://unsloth.ai/docs/basics/text-to-speech-tts-fine-tuning) and [vision](https://unsloth.ai/docs/basics/vision-fine-tuning) model support.
4. Explore our [LLM Tutorials Directory](https://unsloth.ai/docs/models/tutorials-how-to-fine-tune-and-run-llms) to find dedicated guides for each model.
5. Need help with Inference? Read our [Inference & Deployment page](https://unsloth.ai/docs/basics/inference-and-deployment) for details on using vLLM, llama.cpp, Ollama etc.

<div class="align-center">
  <a href="https://unsloth.ai"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
  <a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord.png" width="145"></a>
  <a href="https://unsloth.ai/docs/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a>

  Join Discord if you need help + ⭐️ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐️

  <b>This notebook and all Unsloth notebooks are licensed [LGPL-3.0](https://github.com/unslothai/notebooks?tab=LGPL-3.0-1-ov-file#readme)</b>
</div>